# 7주차 실습. 흐름과 자기회귀 모형으로 정확한 log p(x) 구하기
*Week 7 lab. Exact log p(x) with a flow and an autoregressive model*

**제출하지 않습니다. 채점하지 않습니다.**
*Nothing to submit, nothing graded.*

**이 실습의 목표 한 문장.** 결합층 하나의 숫자를 손으로 확인하고, 층을 쌓으면 log-det 이 층마다 더해진다는 것을 한 줄로 직접 쓴 뒤, 그 줄로 학습한 2차원 흐름이 보류 점마다 정확한 log p(x) 를 주는 것을 확인합니다. 그 점수로 희귀 점을 찾을 때 무엇이 잡히고 무엇이 빠지는지 세고, 자기회귀 모형은 log p(x) 를 조건부확률의 곱으로 구한다는 것을 세 픽셀로 확인합니다.
*Goal in one sentence. Check the numbers of one coupling layer by hand, write the one line that adds up the log-det over a stack of layers, then see that a 2-D flow trained with that line gives an exact log p(x) for every held-out point. Count what that score catches, and what it misses, when we look for rare points, and check on three pixels that an autoregressive model gets log p(x) as a product of conditionals.*

| 절 / Section | 하는 일 / What you do | 시간 / Time |
|---|---|---|
| §0 | 준비. 쉬는 시간에 실행하세요(§4 용 MNIST 내려받기)<br>Setup. Run it during the break (downloads MNIST for §4) | 쉬는 시간 / during the break |
| 몸풀기 / Warm-up | 슬라이드의 2 × 2 예로 야코비안과 행렬식 확인. 빈칸 없음<br>Check the Jacobian and the determinant on the 2 × 2 example from the slides. No blank | 2분 / 2 min |
| §1 | 결합층 하나를 손으로 계산하고 코드로 확인<br>Work out one coupling layer by hand, then check it in code | 7분 / 7 min |
| §2 | 빈칸 1개. 쌓은 층 전체의 log-det<br>The only blank. The log-det of a stack of layers | 6분 / 6 min |
| §3 | 2차원 흐름을 학습하고 보류 점을 채점, 가장 낮은 5% 점검<br>Train a 2-D flow, score held-out points, check the lowest 5% | 10분 / 10 min |
| 세 픽셀 / Three pixels | 자기회귀 모형이 세 픽셀의 log p(x) 를 구하는 계산. 빈칸 없음<br>How an autoregressive model gets log p(x) for three pixels. No blank | 3분 / 3 min |
| §4 | (선택) 이진 MNIST 의 작은 자기회귀 모형. 생성 시간이 D 와 함께 늘어남<br>(Optional) A tiny autoregressive model on binary MNIST. Sampling time grows with D | CPU 약 1분 / about 1 min on a CPU |
| 정리 / Wrap-up | 예측 다시 읽기<br>Re-read your predictions | 2분 / 2 min |

수업 중 실습은 30분입니다. §4 는 시간이 남을 때나 집에서 하세요.
*The lab gets 30 minutes in class. Do §4 if time is left, or at home.*

슬라이드와 짝(7주차 본 덱): 몸풀기 「Two Neurons Form a Vector Map」 · 「Why Do We Need a Matrix?」 · 「Why Does a Determinant Measure Area?」, §1 「How Does RealNVP Make a Layer Invertible?」 · 「Why Is the Coupling Log-Determinant So Easy?」 · 「Optional A · One Coupling Layer, All the Numbers」, §2 「Stretching Spreads Probability Thin」 · 「The Change-of-Variables Formula」 · 「In D Dimensions, the Stretch Is a Determinant」, §3 「Training Is Plain Maximum Likelihood」 · 「Can an Exact p(x) Find Unusual Images?」 · 「Can Images the Flow Never Saw Score Higher?」, 세 픽셀 「How Does an AR Model Score Three Pixels?」 · 「Training Is Plain Maximum Likelihood」 · 「Why Can Training Score Every Pixel at Once?」, §4 「AR Models Turn the Chain Rule Into a Generator」 · 「Masked AR Scoring Is Parallel, Sampling Is Not」, 함정 「Four Ways to Get This Wrong」.
*Matching slides in the Week 7 main deck, section by section, with the same titles.*

**GPU 가 없어도 됩니다.** 2코어 CPU 에서 §3 학습이 10~15초, 나머지 셀은 몇 초 안쪽입니다.
*No GPU needed. On a 2-core CPU the §3 training takes 10 to 15 seconds; every other cell takes a few seconds at most.*

## §0 준비 / Setup
아래 셀을 실행하면 라이브러리를 불러오고, §4 에서 쓸 MNIST 를 내려받습니다(약 12 MB, 30초 안팎). **쉬는 시간에 실행해 두세요.** 내려받기가 실패해도 §1 ~ §3 은 그대로 진행됩니다. 그때 §4 는 라이브러리에 들어 있는 작은 손글씨 숫자(8 × 8)로 대신합니다.
*Run the cell below. It imports the libraries and downloads MNIST for §4 (about 12 MB, around 30 seconds). **Run it during the break.** If the download fails, §1 to §3 still run, and §4 uses a small set of 8 × 8 handwritten digits that ships with a library instead.*

먼저 **파일 → Drive에 사본 저장**을 누르세요. 그러지 않으면 작업 내용이 사라집니다.
*First, File → Save a copy in Drive. Otherwise your work is lost.*

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_num_threads(2)   # 스레드 수 고정. 같은 컴퓨터에서 다시 돌리면 같은 숫자가 나온다 / fix the thread count so a re-run gives the same numbers

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
# Message shown when a TODO is still blank. It means the notebook is fine, not broken.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막힐 때 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace None above with your code and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell just below and continue.\n"
)

GREEN, GOLD, RED, GRAY = "#406551", "#C9971C", "#B04A3A", "#9AA5A0"   # 그림 색 / figure colors

# MNIST 는 §4 에서만 씁니다. 지금 내려받아 둡니다(약 12 MB) / MNIST is used only in §4; download it now (about 12 MB)
try:
    from torchvision import datasets
    mnist_train = datasets.MNIST("data", train=True, download=True)
    mnist_test = datasets.MNIST("data", train=False, download=True)
    print("MNIST 학습 · 시험 이미지 / MNIST training and test images:", len(mnist_train), len(mnist_test))
except Exception as e:                       # 네트워크가 막혀도 §1 ~ §3 은 진행 / §1 to §3 do not need MNIST
    mnist_train = mnist_test = None
    print("MNIST 를 내려받지 못했습니다. §1 ~ §3 은 그대로 진행하세요. §4 는 작은 숫자 이미지로 대신합니다.")
    print("Could not download MNIST. Go on with §1 to §3; §4 will use a small digits set instead.", type(e).__name__)
# 기대값 / Expected   60000 10000

## 몸풀기. 슬라이드의 2 × 2 예 / Warm-up. The 2 × 2 example from the slides
「Two Neurons Form a Vector Map」의 두 뉴런을 사상 하나로 씁니다. 편향과 ReLU 는 빼고 가중치만 씁니다.
*The two neurons from that slide, used as one map, with weights only: no bias and no ReLU.*

$$x_1 = 2z_1 + z_2, \qquad x_2 = z_1 + 2z_2, \qquad x = Wz, \quad W = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$$

**코드를 돌리기 전에** 슬라이드에서 두 답을 찾아 말해 보세요.
*Before running, find two answers on the slides and say them.*

1. 야코비안 J 는? / What is the Jacobian J? (「Why Do We Need a Matrix?」)
2. $\det J$ 는? 그러면 $p(x)$ 와 $p(z)$ 는 어떤 관계인가요? (「Why Does a Determinant Measure Area?」)
   *What is $\det J$, and how are $p(x)$ and $p(z)$ related?*

아래 셀은 컴퓨터가 J 를 직접 구하게 합니다. §1 의 확인 셀이 같은 도구로 4 × 4 표를 구합니다.
*The cell below has the computer find J itself. The check in §1 uses the same tool on a 4 × 4 table.*

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   W  : 두 뉴런의 가중치 표. 1행 = 뉴런 1 의 가중치 (2, 1), 2행 = 뉴런 2 의 가중치 (1, 2)
#        the two neurons' weight table; row 1 = neuron 1's weights (2, 1), row 2 = neuron 2's weights (1, 2)
#   f  : z 를 받아 x = Wz 를 돌려주는 사상 / the map that takes z and returns x = Wz
#   z0 : J 를 재는 점 / the point where J is measured
#
# 참고 / Reference   torch.autograd.functional.jacobian(f, z)
#   점 z 에서, f 의 출력 하나를 입력 하나로 미분한 값(다른 입력은 고정)을 모두 모은 표를 돌려준다.
#   At the point z, returns the table of derivatives of each output of f with respect to each input (the other inputs held fixed).
#   예 / e.g.   f(z) = 3z 이면 / if f(z) = 3z:   jacobian(f, torch.tensor([1.0]))  ->  tensor([[3.]])
# 참고 / Reference   torch.linalg.det(A)
#   정사각 표 A 의 행렬식. 2 × 2 이면 ad − bc / the determinant of a square table A; for 2 × 2 it is ad − bc
#   예 / e.g.   torch.linalg.det(torch.tensor([[1.0, 2.0], [3.0, 4.0]]))  ->  -2   (1 × 4 − 2 × 3)

W = torch.tensor([[2.0, 1.0],
                  [1.0, 2.0]])                     # 슬라이드의 W / the W on the slide

def f(z):
    return W @ z                                   # x = Wz. @ 는 행렬 곱 / @ is the matrix product

z0 = torch.tensor([1.0, 0.0])                      # z = (1, 0) 에서 잰다 / measure at z = (1, 0)
J = torch.autograd.functional.jacobian(f, z0)      # 2 × 2 표 / a 2 × 2 table
det_J = torch.linalg.det(J).item()                 # 행렬식, 숫자 하나 / the determinant, one number
print("J =")
print(J.numpy())
print("det J      =", round(det_J, 4))
print("log|det J| =", round(math.log(abs(det_J)), 4))
# 기대값 / Expected
#   J = [[2. 1.]
#        [1. 2.]]        W 와 같습니다 / the same as W
#   det J      = 3.0     2 × 2 − 1 × 1. 넓이 1 인 정사각형이 넓이 3 이 됩니다 / a square of area 1 becomes area 3
#   log|det J| = 1.0986  ln 3
#   그래서 p(x) = p(z) / 3, 로그로는 log p(x) = log p(z) − 1.0986 입니다(「In D Dimensions, the Stretch Is a Determinant」의 식).
#   So p(x) = p(z) / 3, and in logs log p(x) = log p(z) − 1.0986 (the formula on that slide).
#   z0 를 다른 점으로 바꿔도 J 는 같습니다. 가중치만 쓰는 직선 사상이기 때문입니다.
#   Moving z0 elsewhere gives the same J, because a weights-only map is a straight-line map.
#   §1 의 결합층은 안에 신경망이 있어 J 가 점에 따라 달라집니다.
#   The coupling layer in §1 has networks inside, so its J changes from point to point.

## §1 결합층 하나를 손으로 / One coupling layer by hand
「Optional A · One Coupling Layer, All the Numbers」의 숫자를 그대로 씁니다. 점 하나(숫자 4개)를 두 절반으로 나눕니다.
*We use the numbers on that slide. One point (4 numbers) is split into two halves.*

- 그대로 두는 절반 $x_1 = (0.5,\ -1.0)$, 바뀌는 절반 $x_2 = (1.0,\ 2.0)$
  *the kept half $x_1$ and the changed half $x_2$*
- 신경망이 $x_1$ 을 읽고 낸 값 $s = (0.4,\ -0.3)$, $t = (0.3,\ 0.1)$
  *the values the network returns after reading $x_1$*

결합층의 규칙(「How Does RealNVP Make a Layer Invertible?」). $\odot$ 는 같은 자리끼리 곱하기입니다.
*The coupling rule. $\odot$ multiplies entry by entry.*

$$x_1' = x_1, \qquad x_2' = x_2 \odot e^{s} + t, \qquad \log|\det J| = \textstyle\sum_j s_j$$

**코드를 돌리기 전에** 손이나 계산기로 세 가지를 적으세요. $e^{0.4} = 1.4918$, $e^{-0.3} = 0.7408$ 을 쓰면 됩니다.
*Before running any code, write down three things by hand or with a calculator, using $e^{0.4} = 1.4918$ and $e^{-0.3} = 0.7408$.*

1. $x_2'$ 는? / What is $x_2'$?
2. $\log|\det J|$ 는? / What is $\log|\det J|$?
3. $x_2'$ 에서 $x_2$ 를 되찾는 식은? 신경망을 거꾸로 돌려야 하나요?
   *Which formula gets $x_2$ back from $x_2'$? Do you have to run the network backward?*

In [ ]:
# 여기에 손으로 구한 값을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your hand-computed values here. This cell does not need to be run.
my_x2_new = ""
my_logdet = ""
my_inverse = ""

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   x1, x2 : 그대로 두는 절반, 바뀌는 절반 / the kept half, the changed half
#   s, t   : 신경망이 x1 을 읽고 낸 크기 조절(s)과 이동(t). 여기서는 슬라이드의 숫자를 그대로 넣는다
#            the scale (s) and shift (t) the network returns after reading x1; here we type in the slide's numbers
x1 = torch.tensor([0.5, -1.0])
x2 = torch.tensor([1.0, 2.0])
s = torch.tensor([0.4, -0.3])
t = torch.tensor([0.3, 0.1])

x2_new = x2 * torch.exp(s) + t          # 앞으로: 늘이고(× e^s) 옮긴다(+ t) / forward: scale (× e^s), then shift (+ t)
logdet = s.sum()                        # 이 층의 log|det J| = s 의 합 / this layer's log|det J| = the sum of s
x2_back = (x2_new - t) * torch.exp(-s)  # 거꾸로: 옮긴 만큼 빼고, 늘인 만큼 나눈다 / backward: undo the shift, then undo the scale

print("x2'                      =", x2_new.numpy().round(4))
print("log|det J|               =", round(logdet.item(), 4))
print("x2 되찾기 / x2 recovered =", x2_back.numpy().round(4))
# 기대값 / Expected
#   x2'          = [1.7918 1.5816]     ( 1.0 × 1.4918 + 0.3,  2.0 × 0.7408 + 0.1 )
#   log|det J|   = 0.1                 ( 0.4 + (−0.3) )
#   x2 recovered = [1. 2.]
#   되찾을 때도 같은 s, t 를 씁니다. x1 이 바뀌지 않았으니 신경망을 x1 에 다시 돌리면 같은 s, t 가 나옵니다.
#   신경망을 거꾸로 돌릴 필요가 없습니다.
#   Getting x2 back uses the same s and t: x1 did not change, so running the network on x1 again gives the same s and t.
#   The network never has to run backward.

### §1 확인. 행렬을 딱 한 번 만들어 보기 / Check. Build the matrix just once
슬라이드는 「행렬을 만들지 않고 신경망의 출력에서 log-det 을 읽는다」고 했습니다. 여기서는 확인용으로 딱 한 번 그 행렬을 만들어 봅니다. 출력 하나를 입력 하나로 미분한 값(다른 입력은 고정)을 모은 4 × 4 표, 곧 야코비 행렬 J 입니다. 이번에는 s, t 를 숫자로 넣지 않고 작은 신경망이 $x_1$ 에서 계산하게 합니다. 그래서 s 는 위의 (0.4, −0.3) 이 아니고, log-det 도 0.1 이 아닌 다른 값이 나옵니다. 볼 것은 그 숫자가 아니라 행렬로 구한 값과 s 의 합이 같은지입니다. 행렬은 몸풀기에서 쓴 `jacobian` 으로 구합니다.
*The slide says the log-det is read off the network's output, with no matrix formed. Here, only to check, we build that matrix just once: the 4 × 4 table of derivatives of each output with respect to each input (the other inputs held fixed), the Jacobian J. This time s and t are not typed in; a small network computes them from $x_1$. So s is not the (0.4, −0.3) above, and the log-det comes out as a number other than 0.1. What matters is not that number but whether the value from the matrix equals the sum of s. The matrix comes from `jacobian`, as in the warm-up.*

In [ ]:
torch.manual_seed(0)                                  # 신경망 가중치 고정 / fix the network's weights
net = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 4))   # x1(숫자 2개)을 읽어 s, t(숫자 2개씩)를 낸다 / reads x1 (2 numbers), returns s and t (2 numbers each)

def coupling(x):
    """점 하나 [x1, x2](숫자 4개)를 결합층에 통과시켜 [x1', x2'] 를 돌려준다.
    Pass one point [x1, x2] (4 numbers) through a coupling layer and return [x1', x2']."""
    x1, x2 = x[:2], x[2:]                             # 앞 둘은 그대로, 뒤 둘은 바뀐다 / the first two are kept, the last two change
    h = net(x1)                                       # 신경망은 x1 만 읽는다 / the network reads only x1
    s, t = torch.tanh(h[:2]), h[2:]                   # s 는 −1 과 1 사이로 묶는다 / s is kept between −1 and 1
    return torch.cat([x1, x2 * torch.exp(s) + t])     # [x1', x2'] = [x1, x2 ⊙ e^s + t]

x = torch.tensor([0.5, -1.0, 1.0, 2.0])               # 위와 같은 점 / the same point as above
J = torch.autograd.functional.jacobian(coupling, x)   # J[i, j] = 출력 i 를 입력 j 로 미분 / J[i, j] = derivative of output i with respect to input j
print("J =")
print(J.numpy().round(3))

s_net = torch.tanh(net(x[:2])[:2])                    # 이 신경망이 낸 s / the s this network produced
print("이 신경망의 s / s from this network      :", s_net.detach().numpy().round(4), "   e^s =", torch.exp(s_net).detach().numpy().round(3))

# 참고 / Reference   torch.linalg.slogdet(A).logabsdet
#   log|det A|. 몸풀기의 det 에 절댓값과 로그를 씌운 값 / log|det A|: the warm-up's det with an absolute value and a log taken
print("행렬로 구한 log|det J| / from the matrix :", round(torch.linalg.slogdet(J).logabsdet.item(), 4))
print("s 의 합 / sum of s                      :", round(s_net.sum().item(), 4))
# 기대값 / Expected
#   J 의 오른쪽 위 2 × 2 칸은 모두 0, 오른쪽 아래 2 × 2 칸은 대각선(e^s 두 개)만 0 이 아닙니다.
#   The top-right 2 × 2 block of J is all zeros; in the bottom-right 2 × 2 block only the diagonal (the two e^s) is nonzero.
#   s = [-0.2572 -0.3768],  e^s = [0.773 0.686]. e^s 가 J 의 오른쪽 아래 대각선과 같습니다 / e^s matches that diagonal of J
#   마지막 두 줄의 값이 같습니다 / the last two lines agree:  -0.634
#   0.1 이 아니어도 정상입니다. 이번 s 는 슬라이드의 숫자가 아니라 이 신경망이 정한 값입니다.
#   Not 0.1, and that is expected: this time s comes from this network, not from the slide.

**왜 같은가.** $x_1' = x_1$ 이라 $x_2$ 를 바꿔도 $x_1'$ 은 그대로입니다. 그래서 J 의 오른쪽 위가 0 이고, J 는 삼각 모양이 됩니다. 삼각 행렬의 행렬식은 대각선의 곱이고, 대각선은 $1, 1, e^{s_1}, e^{s_2}$ 이므로 $\log|\det J| = s_1 + s_2$ 입니다. 왼쪽 아래 칸($x_2'$ 를 $x_1$ 로 미분한 값)은 신경망 때문에 복잡하지만 행렬식에 들어가지 않습니다. 그래서 s, t 가 아무리 복잡한 신경망이어도 log-det 은 대각선만으로 계산됩니다. s, t 가 가역일 필요가 없는 것은 다른 이유입니다. 거꾸로 갈 때도 x1 에서 s, t 를 다시 계산할 뿐 뒤집지 않습니다(「s and t need not be invertible: any network works.」).
*Why they agree. $x_1' = x_1$, so changing $x_2$ never changes $x_1'$. That makes the top-right block zero, so J is triangular. A triangular matrix's determinant is the product of its diagonal, here $1, 1, e^{s_1}, e^{s_2}$, so $\log|\det J| = s_1 + s_2$. The bottom-left block (how $x_2'$ changes with $x_1$) is complicated because of the network, but it never enters the determinant. So however complex s and t are, the log-det needs only the diagonal. That s and t need not be invertible is a separate fact: going backward, we recompute s and t from x1 and never invert them.*

## §2 빈칸 하나. 쌓은 층 전체의 log-det / The only blank. The log-det of a stack
결합층 하나는 한쪽 절반만 바꿉니다. 그래서 층을 여러 개 쌓고, 그대로 두는 절반을 층마다 번갈아 바꿉니다.
*One coupling layer changes only one half. So we stack several layers and swap which half is kept from layer to layer.*

층을 쌓으면 log-det 은 어떻게 될까요? 1차원에서 먼저 봅니다. 두 사상을 이어 붙인 $x = f_2(f_1(z))$ 의 기울기는 연쇄법칙으로 $f_2' \cdot f_1'$, 곧 곱입니다. 늘이는 비율이 곱해지므로 로그를 씌우면 더해집니다.
*What happens to the log-det when layers are stacked? Look at one dimension first. For $x = f_2(f_1(z))$ the chain rule gives the slope $f_2' \cdot f_1'$, a product. Stretch factors multiply, so their logs add.*

$$\log|(f_2 \circ f_1)'| = \log|f_2'| + \log|f_1'|$$

예. 2배로 늘인 다음 3배로 늘이면 모두 6배이고, 로그로는 $\ln 2 + \ln 3 = \ln 6$ 입니다(「Stretching Spreads Probability Thin」의 $\ln 2$ 를 두 번 쓴 셈).
*Example. Stretch by 2, then by 3: 6 times in all, and in logs $\ln 2 + \ln 3 = \ln 6$.*

D 차원에서도 같습니다(결과만). 이어 붙인 사상의 행렬은 층마다의 행렬의 곱이고, 선형대수의 $\det(AB) = \det(A)\det(B)$ 에 따라 행렬식이 곱해지므로 로그는 더해집니다. §1 에서 본 대로 결합층 하나의 log-det 은 그 층 s 의 합이므로, K 층을 쌓은 흐름은 다음과 같습니다.
*The same holds in D dimensions (result only). The matrix of the stacked map is the product of the layers' matrices, and $\det(AB) = \det(A)\det(B)$, so the logs add. One coupling layer's log-det is the sum of its s (§1), so for K stacked layers:*

$$\log|\det J_{\text{stack}}| = \sum_{k=1}^{K} \sum_j s^{(k)}_j$$

**빈칸 하나.** 층마다의 s 를 받아 이 합을 점마다 돌려주는 함수입니다. §3 의 흐름이 이 함수를 그대로 씁니다.
*The only blank. A function that takes each layer's s and returns this sum for every point. The flow in §3 uses this very function.*

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   s_list    : 층마다 하나씩 s 를 담은 목록. 각 s 의 크기는 (점 수, 그 층에서 바뀐 좌표 수)
#               a list with one s per layer; each s has shape (number of points, coordinates changed in that layer)
#   total     : 지금까지 지나온 층들의 log|det| 합, 점마다 하나. 크기 (점 수,)
#               the log|det| summed over the layers so far, one per point, shape (number of points,)
#   new_total : total 에 이 층의 log|det| 를 더한 것   <- 여기가 빈칸
#               total plus this layer's log|det|   <- this is the blank
#
# 참고 / Reference   a.sum(dim=1)
#   2차원 텐서 a 의 행마다 칸을 더해, 행마다 숫자 하나를 돌려준다. dim=1 을 빼면 모든 칸을 더한 숫자 하나가 된다.
#   Adds the entries of each row of a 2-D tensor a, one number per row. Without dim=1 you get one number: the sum of every entry.
#   예 / e.g.   torch.tensor([[1., 2.], [3., 4.]]).sum(dim=1)  ->  tensor([3., 7.])

def stack_logdet(s_list):
    """쌓은 층 전체의 log|det J| 를 점마다 돌려준다(층마다 s 의 합을, 층에 걸쳐 다시 더한 것).
    Returns log|det J| of the whole stack for every point (each layer's sum of s, added over the layers)."""
    total = torch.zeros(s_list[0].shape[0])   # 점마다 0 에서 시작 / start at 0 for every point
    for s in s_list:                          # 층마다 한 번 / once per layer
        # TODO. total 에 이 층의 log|det J| 를 더한 값을 new_total 에 넣으세요.
        #       한 층의 log|det J| 는 §1 에서 무엇이었나요? 점마다 따로 구해야 합니다. 한 줄입니다.
        # TODO. Put total plus this layer's log|det J| into new_total.
        #       What was one layer's log|det J| in §1? Work it out separately for each point. One line.
        new_total = None

        if new_total is None:
            raise NotImplementedError(BLANK_MSG)

        total = new_total                     # 다음 층으로 넘긴다 / carry it to the next layer
    return total

# 자가검증 / Check your work
# 점 두 개, 층 두 개. 첫 점의 첫 층은 §1 의 s = (0.4, −0.3) 입니다.
# Two points, two layers. The first point's first layer is the s = (0.4, −0.3) of §1.
s_layer1 = torch.tensor([[0.4, -0.3],
                         [0.0,  0.0]])        # 첫 층: 점 1, 점 2 / layer 1: point 1, point 2
s_layer2 = torch.tensor([[0.2,  0.5],
                         [1.0, -1.0]])        # 둘째 층: 점 1, 점 2 / layer 2: point 1, point 2
print(stack_logdet([s_layer1, s_layer2]))
# 기대값 / Expected   tensor([0.8000, 0.0000])   ( 점 1 / point 1: 0.1 + 0.7,  점 2 / point 2: 0.0 + 0.0 )
#   tensor([0.7000, 0.0000]) 이면 total 을 더하지 않아 마지막 층만 남았습니다.
#   tensor([0.7000, 0.0000]) means total was not added, so only the last layer is left.
#   tensor([0.8000, 0.8000]) 이면 dim=1 을 빼서 두 점의 s 를 한데 더했습니다. 점마다 따로 더해야 합니다.
#   tensor([0.8000, 0.8000]) means dim=1 was dropped, so the two points' s were lumped together. Add per point.
#   tensor([5.1028, 5.0862]) 이면 log 대신 늘이는 비율 e^s 를 더했습니다. log|det| 에는 s 자체가 들어갑니다.
#   tensor([5.1028, 5.0862]) means you added the stretch factors e^s instead of their logs. The log|det| takes s itself.

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요. 이미 채웠다면 실행하지 마세요(내 함수를 덮어씁니다) / Only if stuck. If you already filled in the blank, skip it: it overwrites your function { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

# 다음 이름을 씁니다 / Names used below
#   s_list    : 층마다 하나씩 s 를 담은 목록. 각 s 의 크기는 (점 수, 그 층에서 바뀐 좌표 수)
#               a list with one s per layer; each s has shape (number of points, coordinates changed in that layer)
#   total     : 지금까지 지나온 층들의 log|det| 합, 점마다 하나. 크기 (점 수,)
#               the log|det| summed over the layers so far, one per point, shape (number of points,)
#   new_total : total 에 이 층의 log|det| 를 더한 것   <- 여기가 빈칸
#               total plus this layer's log|det|   <- this is the blank
#
# 참고 / Reference   a.sum(dim=1)
#   2차원 텐서 a 의 행마다 칸을 더해, 행마다 숫자 하나를 돌려준다. dim=1 을 빼면 모든 칸을 더한 숫자 하나가 된다.
#   Adds the entries of each row of a 2-D tensor a, one number per row. Without dim=1 you get one number: the sum of every entry.
#   예 / e.g.   torch.tensor([[1., 2.], [3., 4.]]).sum(dim=1)  ->  tensor([3., 7.])

def stack_logdet(s_list):
    """쌓은 층 전체의 log|det J| 를 점마다 돌려준다(층마다 s 의 합을, 층에 걸쳐 다시 더한 것).
    Returns log|det J| of the whole stack for every point (each layer's sum of s, added over the layers)."""
    total = torch.zeros(s_list[0].shape[0])   # 점마다 0 에서 시작 / start at 0 for every point
    for s in s_list:                          # 층마다 한 번 / once per layer
        new_total = total + s.sum(dim=1)      # 빈칸이었던 줄. 이 층의 log|det J| = 점마다 s 의 합. 층마다 더한다
                                              # the blank. This layer's log|det J| = the sum of s for each point, added layer by layer

        if new_total is None:
            raise NotImplementedError(BLANK_MSG)

        total = new_total                     # 다음 층으로 넘긴다 / carry it to the next layer
    return total

# 자가검증 / Check your work
# 점 두 개, 층 두 개. 첫 점의 첫 층은 §1 의 s = (0.4, −0.3) 입니다.
# Two points, two layers. The first point's first layer is the s = (0.4, −0.3) of §1.
s_layer1 = torch.tensor([[0.4, -0.3],
                         [0.0,  0.0]])        # 첫 층: 점 1, 점 2 / layer 1: point 1, point 2
s_layer2 = torch.tensor([[0.2,  0.5],
                         [1.0, -1.0]])        # 둘째 층: 점 1, 점 2 / layer 2: point 1, point 2
print(stack_logdet([s_layer1, s_layer2]))
# 기대값 / Expected   tensor([0.8000, 0.0000])   ( 점 1 / point 1: 0.1 + 0.7,  점 2 / point 2: 0.0 + 0.0 )
#   tensor([0.7000, 0.0000]) 이면 total 을 더하지 않아 마지막 층만 남았습니다.
#   tensor([0.7000, 0.0000]) means total was not added, so only the last layer is left.
#   tensor([0.8000, 0.8000]) 이면 dim=1 을 빼서 두 점의 s 를 한데 더했습니다. 점마다 따로 더해야 합니다.
#   tensor([0.8000, 0.8000]) means dim=1 was dropped, so the two points' s were lumped together. Add per point.
#   tensor([5.1028, 5.0862]) 이면 log 대신 늘이는 비율 e^s 를 더했습니다. log|det| 에는 s 자체가 들어갑니다.
#   tensor([5.1028, 5.0862]) means you added the stretch factors e^s instead of their logs. The log|det| takes s itself.

## §3 2차원 흐름으로 보류 점 채점하기 / Score held-out points with a 2-D flow
흔한 데이터는 두 반달(two moons) 위의 점 2,000개입니다. 결합층 8개를 쌓은 흐름을 이 점들로 학습합니다. 학습은 「Training Is Plain Maximum Likelihood」 그대로, 학습 점들의 평균 −log p(x) 를 줄이는 것입니다.
*The common data are 2,000 points on two half-moons. We train a flow of 8 stacked coupling layers on them. Training is plain maximum likelihood: make the average −log p(x) of the training points small.*

점 하나의 log p(x) 는 「The Change-of-Variables Formula」대로 구합니다. 점 x 를 마지막 층부터 거꾸로 통과시켜 z 를 얻고(채점 방향 x → z), 층마다 나온 s 를 §2 의 `stack_logdet` 로 더해 뺍니다. 거꾸로 갈 때도 층마다 s 는 앞으로 갈 때와 같으므로(§1), 이 합은 생성 방향의 사상 f 가 공간을 늘인 정도 $\log|\det \partial f / \partial z|$ 입니다.
*For one point: run x backward through the layers, last layer first, to get z (the scoring direction, x → z); add up each layer's s with your `stack_logdet` from §2 and subtract it. Going backward, each layer's s is the same as going forward (§1), so this sum is how much the generating map f stretches space, $\log|\det \partial f / \partial z|$.*

$$\log p(x) = \log p(z) - \log|\det \partial f / \partial z|, \qquad p(z) = \text{2차원 표준정규 / the 2-D standard normal}$$

**코드를 돌리기 전에** 두 가지를 예측해 적으세요. / **Before running**, write down two predictions.

1. 보류 점 500개를 흐름과, 평균 · 공분산만 맞춘 정규분포 하나(비교 기준)로 채점합니다. 평균 log p(x) 는 어느 쪽이 높을까요?
   *We score 500 held-out points with the flow and with a single Gaussian fitted by its mean and covariance (the baseline). Which gets the higher average log p(x)?*
2. 보류 점에 희귀 점 25개를 섞습니다. 15개는 반달에서 떨어진 곳에, 10개는 위쪽 반달의 꼭대기, 흔한 점들 바로 위에 몰려 있습니다. log p(x) 가 가장 낮은 5% 에 희귀 점이 몇 개씩 들어갈까요?
   *We mix 25 rare points into the held-out set: 15 sit away from the moons, 10 are bunched at the top of the upper moon, right on top of the common points. How many of each group land in the lowest 5% of log p(x)?*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your predictions here. This cell does not need to be run.
my_prediction_1 = ""
my_prediction_2 = ""

In [ ]:
def sample_moons(n, rng, noise=0.08):
    """두 반달 위의 점 n 개. 위쪽 반달은 반지름 1 인 원의 위 절반이고, 아래쪽은 그것을 뒤집어 오른쪽 아래로 옮긴 것이다.
    n points on two half-moons. The upper moon is the top half of a circle of radius 1; the lower one is that half flipped and moved down and to the right."""
    n_up = n // 2
    n_low = n - n_up
    a = rng.uniform(0, np.pi, n_up)                    # 위쪽 반달의 각도 / angles on the upper moon
    b = rng.uniform(0, np.pi, n_low)                   # 아래쪽 반달의 각도 / angles on the lower moon
    upper = np.stack([np.cos(a), np.sin(a)], axis=1)
    lower = np.stack([1 - np.cos(b), 0.5 - np.sin(b)], axis=1)
    x = np.concatenate([upper, lower]) + noise * rng.standard_normal((n, 2))   # 작은 잡음을 더한다 / add a little noise
    return x[rng.permutation(n)]                       # 두 반달을 섞는다 / shuffle the two moons together

rng = np.random.default_rng(7)                         # 데이터 난수 고정 / fix the data's randomness
x_train = sample_moons(2000, rng)                      # 학습 점 / training points
x_test = sample_moons(500, rng)                        # 보류 점. 학습에 쓰지 않는다 / held-out points, never used in training
X_train = torch.tensor(x_train, dtype=torch.float32)
X_test = torch.tensor(x_test, dtype=torch.float32)
print("학습 · 보류 점 / training and held-out points:", tuple(X_train.shape), tuple(X_test.shape))
# 기대값 / Expected   (2000, 2) (500, 2)

아래 두 클래스가 흐름 전체입니다. `Coupling` 은 §1 의 결합층과 같은 규칙을 2차원에 씁니다. 좌표가 둘뿐이라 「절반」은 좌표 하나입니다. `Flow` 는 결합층 8개를 쌓고, 그대로 두는 좌표를 층마다 번갈아 바꿉니다. `log_prob` 의 마지막 줄이 위의 식이고, 거기서 여러분의 `stack_logdet` 을 부릅니다.
*The two classes below are the whole flow. `Coupling` applies the rule of §1 in two dimensions; with only two coordinates, a "half" is one coordinate. `Flow` stacks 8 coupling layers and alternates the kept coordinate. The last line of `log_prob` is the formula above, and it calls your `stack_logdet`.*

In [ ]:
class Coupling(nn.Module):
    """2차원 결합층 하나. 좌표 하나(keep)는 그대로 두고, 그 값을 읽은 작은 신경망이 낸 s, t 로 다른 좌표를 늘이고 옮긴다.
    One 2-D coupling layer. Keep one coordinate; a small network reads it and returns s and t, which scale and shift the other coordinate."""
    def __init__(self, keep, hidden=64):
        super().__init__()
        self.keep = keep                                          # 그대로 두는 좌표 번호, 0 또는 1 / index of the kept coordinate, 0 or 1
        self.net = nn.Sequential(nn.Linear(1, hidden), nn.Tanh(),
                                 nn.Linear(hidden, hidden), nn.Tanh(),
                                 nn.Linear(hidden, 2))            # 출력 두 개 = s 와 t / two outputs = s and t

    def s_and_t(self, kept):
        h = self.net(kept)
        return torch.tanh(h[:, :1]), h[:, 1:]                     # s 는 −1 과 1 사이로 묶어 학습이 튀지 않게 / s kept in (−1, 1) so training stays calm

    def split(self, x):
        return x[:, self.keep:self.keep + 1], x[:, 1 - self.keep:2 - self.keep]   # (그대로 두는 좌표, 바뀌는 좌표) / (kept, changed)

    def join(self, kept, changed):
        return torch.cat([kept, changed], dim=1) if self.keep == 0 else torch.cat([changed, kept], dim=1)

    def forward(self, x):                                         # 생성 방향 z → x / generating direction z → x
        x1, x2 = self.split(x)
        s, t = self.s_and_t(x1)
        return self.join(x1, x2 * torch.exp(s) + t), s            # x2' = x2 ⊙ e^s + t

    def inverse(self, y):                                         # 채점 방향 x → z / scoring direction x → z
        y1, y2 = self.split(y)
        s, t = self.s_and_t(y1)                                   # 같은 s, t. 그대로 둔 좌표는 바뀌지 않았다 / the same s, t: the kept coordinate did not change
        return self.join(y1, (y2 - t) * torch.exp(-s)), s         # x2 = (x2' − t) ⊙ e^(−s)


class Flow(nn.Module):
    """결합층 K 개를 쌓은 흐름. 그대로 두는 좌표를 층마다 번갈아 바꾼다.
    A flow of K stacked coupling layers, alternating the kept coordinate."""
    def __init__(self, K=8):
        super().__init__()
        self.layers = nn.ModuleList([Coupling(keep=k % 2) for k in range(K)])

    def log_prob(self, x):
        z, s_list = x, []
        for layer in reversed(self.layers):                       # 채점은 마지막 층부터 거꾸로 / scoring runs the layers backward, last first
            z, s = layer.inverse(z)
            s_list.append(s)                                      # 층마다 s 를 모은다 / collect each layer's s
        log_pz = -0.5 * (z ** 2).sum(dim=1) - math.log(2 * math.pi)   # 2차원 표준정규의 log 밀도 / log density of the 2-D standard normal
        return log_pz - stack_logdet(s_list)                      # log p(x) = log p(z) − log|det ∂f/∂z|

    @torch.no_grad()
    def sample(self, n, gen):
        z = torch.randn(n, 2, generator=gen)                      # 쉬운 잡음에서 / start from easy noise
        for layer in self.layers:                                 # 생성은 첫 층부터 앞으로 / generation runs the layers forward
            z, _ = layer(z)
        return z

In [ ]:
torch.manual_seed(0)                                     # 신경망 초기값 고정 / fix the initial weights
flow = Flow(K=8)
opt = torch.optim.Adam(flow.parameters(), lr=2e-3)       # 가중치를 고치는 규칙(경사하강법의 한 종류) / the update rule, a kind of gradient descent
gen = torch.Generator().manual_seed(1)                   # 미니배치를 뽑는 난수 고정 / fix the minibatch draws
losses = []
t0 = time.time()
for step in range(1, 2001):
    idx = torch.randint(0, len(X_train), (256,), generator=gen)   # 학습 점 256개를 무작위로 / 256 random training points
    loss = -flow.log_prob(X_train[idx]).mean()           # 평균 −log p(x). 이것이 손실 전부다 / the average −log p(x); this is the whole loss
    opt.zero_grad()                                      # 지난 걸음의 기울기를 지운다 / clear last step's gradients
    loss.backward()                                      # 기울기 계산 / compute the gradients
    opt.step()                                           # 가중치 한 걸음 / one update of the weights
    losses.append(loss.item())
    if step in (1, 500, 1000, 2000):
        print(f"step {step:5,d}:  평균 −log p(x) / average −log p(x) = {loss.item():.3f}")
print(f"걸린 시간 / time: {time.time() - t0:.0f} s")
# 기대값 / Expected   (같은 런타임에서 다시 실행하면 같은 숫자 / a re-run in the same runtime gives the same numbers)
#   step     1: 2.580
#   step   500: 1.020
#   step 1,000: 0.835
#   step 2,000: 0.806
#   시간: 이 노트북을 만든 2코어 CPU 에서 10~15초 / time: 10 to 15 s on the 2-core CPUs used to build this notebook
#   다른 컴퓨터 · 다른 PyTorch 판에서는 소수점 아래가 조금 다를 수 있습니다.
#   On another computer or PyTorch version the decimals can differ slightly.

plt.figure(figsize=(5, 2.6))
plt.plot(losses, color=GREEN, lw=0.8)
plt.xlabel("step"); plt.ylabel("average −log p(x)"); plt.title("Training loss of the flow", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
def gaussian_log_prob(x, mean, cov):
    """평균 mean, 공분산 cov 인 2차원 정규분포의 log 밀도. 식 그대로 계산한다.
    Log density of a 2-D normal with this mean and covariance, computed straight from the formula:
    log N(x; m, C) = −½ (x − m)ᵀ C⁻¹ (x − m) − ½ log det C − log 2π"""
    d = x - mean
    quad = (d @ torch.linalg.inv(cov) * d).sum(dim=1)   # (x − m)ᵀ C⁻¹ (x − m), 점마다 하나 / one per point
    return -0.5 * quad - 0.5 * torch.logdet(cov) - math.log(2 * math.pi)

mean = X_train.mean(dim=0)                               # 학습 점의 평균 / mean of the training points
cov = torch.cov(X_train.T, correction=0)                 # 학습 점의 공분산 / covariance of the training points
with torch.no_grad():                                    # 채점만 하므로 기울기가 필요 없다 / scoring only, no gradients needed
    lp_flow = flow.log_prob(X_test)                      # 보류 점 500개의 log p(x) / log p(x) of the 500 held-out points
lp_gauss = gaussian_log_prob(X_test, mean, cov)
print(f"보류 점 평균 log p(x) / held-out average log p(x)")
print(f"  흐름 / flow                 : {lp_flow.mean():.3f}")
print(f"  정규분포 하나 / one Gaussian : {lp_gauss.mean():.3f}")
# 기대값 / Expected
#   flow         : -0.887
#   one Gaussian : -1.911
#   흐름이 높습니다. 두 값은 같은 보류 점, 같은 좌표에서 잰 정확한 log p(x) 라 그대로 비교할 수 있습니다
#   (「When Do You Need the Number Itself?」의 비교 카드). 학습 점으로 잰 값과 비교하면 안 됩니다.
#   The flow is higher. Both are exact log p(x) on the same held-out points in the same coordinates, so they compare directly.
#   Never compare against a value measured on the training points.

In [ ]:
# 밀도 p(x) 를 격자 위에서 그린다 / Draw the density p(x) on a grid
gx, gy = np.meshgrid(np.linspace(-1.6, 2.6, 160), np.linspace(-1.1, 1.6, 110))
grid = torch.tensor(np.stack([gx.ravel(), gy.ravel()], axis=1), dtype=torch.float32)
with torch.no_grad():
    dens = flow.log_prob(grid).exp().numpy().reshape(gx.shape)   # p(x) = exp(log p(x))
    x_gen = flow.sample(1000, torch.Generator().manual_seed(3)).numpy()   # 흐름에서 뽑은 점 1,000개 / 1,000 samples from the flow

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].scatter(x_train[:, 0], x_train[:, 1], s=2, c=GREEN, alpha=0.5)
axes[0].set_title("Training data (2,000 points)", fontsize=10)
axes[1].contourf(gx, gy, dens, levels=20, cmap="Greens")
axes[1].scatter(x_test[:, 0], x_test[:, 1], s=2, c=GOLD, alpha=0.7)
axes[1].set_title("Flow density p(x), held-out points on top", fontsize=10)
axes[2].scatter(x_gen[:, 0], x_gen[:, 1], s=2, c=GOLD, alpha=0.6)
axes[2].set_title("1,000 samples from the flow (z → x)", fontsize=10)
for ax in axes:
    ax.set_xlim(-1.6, 2.6); ax.set_ylim(-1.1, 1.6); ax.set_aspect("equal")
plt.tight_layout(); plt.show()
# 기대값 / Expected   가운데 그림의 진한 곳이 두 반달을 따라갑니다. 오른쪽 표본도 대부분 두 반달 위에 있습니다.
#   The dark region in the middle panel follows the two moons; most samples on the right lie on the moons too.

### 가장 낮은 5% 점검 / Check the lowest 5%
「Can an Exact p(x) Find Unusual Images?」의 세 단계를 2차원에서 그대로 합니다. 흔한 점으로만 학습한 흐름으로 ① 보류 점을 채점하고 ② log p(x) 가 낮은 순서로 줄 세운 뒤 ③ 가장 낮은 5% 에 희귀 점이 몇 개 들어가는지 셉니다.
*The three steps of that slide, in 2-D. With the flow trained on common points only, ① score the held-out points, ② rank them from the lowest log p(x), ③ count how many rare points land in the lowest 5%.*

희귀 점은 두 무리입니다. **away** 15개는 반달에서 떨어진 왼쪽 아래 (−0.6, −0.6) 근처에, **inside** 10개는 위쪽 반달의 꼭대기 (0, 1) 근처, 흔한 점들이 몰린 바로 그 자리에 있습니다. 이름의 inside 는 「흔한 데이터 안」이라는 뜻입니다. 둘 다 흔한 점과 다른 무리(다른 클래스)라고 가정합니다.
*Two groups of rare points. **away**: 15 points down and to the left near (−0.6, −0.6), away from the moons. **inside**: 10 points near the top of the upper moon, (0, 1), right where the common points crowd. The name means inside the common data. Assume both come from a different group (a different class) than the common points.*

In [ ]:
rng_rare = np.random.default_rng(11)                     # 희귀 점 난수 고정 / fix the rare points' randomness
rare_away = np.array([-0.6, -0.6]) + 0.08 * rng_rare.standard_normal((15, 2))    # 반달에서 떨어진 희귀 점 / rare points away from the moons
rare_inside = np.array([0.0, 1.0]) + 0.05 * rng_rare.standard_normal((10, 2))    # 위쪽 반달 꼭대기, 흔한 점들 위의 희귀 점 / rare points at the top of the upper moon, on top of the common points
x_all = np.concatenate([x_test, rare_away, rare_inside])
group = np.array(["common"] * 500 + ["away"] * 15 + ["inside"] * 10)              # 점마다 어느 무리인지 / which group each point is in
with torch.no_grad():
    lp_all = flow.log_prob(torch.tensor(x_all, dtype=torch.float32)).numpy()      # ① 채점 / ① score

# 참고 / Reference   np.argsort(a)
#   a 를 작은 값부터 줄 세웠을 때의 번호 순서를 돌려준다. 예 / e.g.   np.argsort([3., 1., 2.])  ->  [1, 2, 0]
k = int(0.05 * len(x_all))                               # 가장 낮은 5% 의 개수 / the size of the lowest 5%
lowest = np.argsort(lp_all)[:k]                          # ② 낮은 순서로 줄 세워 앞에서 k 개 / ② rank from the lowest, keep the first k
print(f"가장 낮은 5% = {k}개 / the lowest 5% = {k} points")
for g in ["away", "inside", "common"]:                   # ③ 무리마다 센다 / ③ count per group
    n_in = int((group[lowest] == g).sum())
    print(f"  {g:7s}: {n_in:3d} / {int((group == g).sum()):3d}    log p(x) 중앙값 / median log p(x) = {np.median(lp_all[group == g]):7.2f}")
# 기대값 / Expected
#   가장 낮은 5% = 26개 / the lowest 5% = 26 points
#   away   : 15 / 15    median -47.65
#   inside : 0 / 10    median -0.37
#   common : 11 / 500   median -0.61

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.contourf(gx, gy, dens, levels=20, cmap="Greens", alpha=0.6)
for g, c, mk in [("common", GRAY, "o"), ("away", RED, "s"), ("inside", GOLD, "D")]:
    m = group == g
    ax.scatter(x_all[m, 0], x_all[m, 1], s=8 if g == "common" else 18, c=c, marker=mk, label=g, alpha=0.8)
ax.scatter(x_all[lowest, 0], x_all[lowest, 1], s=70, facecolors="none", edgecolors="black", lw=0.8, label="lowest 5%")
ax.set_xlim(-1.6, 2.6); ax.set_ylim(-1.1, 1.6); ax.set_aspect("equal")
ax.legend(fontsize=8, loc="upper right"); ax.set_title("Which points land in the lowest 5% of log p(x)?", fontsize=10)
plt.tight_layout(); plt.show()

**무엇을 보았나.** 흐름은 자기가 학습한 밀도 p(x) 를 정확히 계산합니다. 그 밀도는 흔한 학습 점들이 어디에 몰려 있는지를 배운 것이고, 실제 데이터의 분포를 그대로 옮겼다는 보장은 없습니다. 흔한 데이터가 없는 곳의 희귀 점(away)은 밀도가 낮아 가장 낮은 5% 에 들어갑니다. inside 는 흔한 점들이 몰린 바로 그 자리에 있어 밀도가 높고, 그래서 빠집니다. 흐름이 보는 것은 점의 자리뿐이고, 그 점이 어느 무리에서 왔는지는 x 에 들어 있지 않습니다. **점수가 높다는 것은 「흔한 데이터와 같은 클래스」라는 뜻이 아닙니다.** 희귀 클래스를 찾는 데 이 점수를 쓸지는 실제 희귀 사례가 든 보류 데이터에서 위처럼 세어 보고 정합니다.
*What did we see. The flow computes its own learned density p(x) exactly. That density was learned from where the common training points crowd together; nothing guarantees it copies the real data distribution. Rare points where no common data lives (away) get a low density and land in the lowest 5%. The inside points sit right where the common points crowd, so their density is high and they are missed. The flow sees only where a point is; which group the point came from is not part of x. **A high score does not mean "the same class as the common data".** Whether to use this score to find a rare class is decided by counting, as above, on held-out data that contains real rare cases.*

inside 의 중앙값(−0.37)이 흔한 점(−0.61)보다도 높은 것은 inside 가 흔한 점보다 덜 퍼져(잡음 크기 0.05 대 0.08) 반달의 능선, 곧 밀도가 가장 높은 선 가까이에 모여 있기 때문입니다. 이미지에서는 더 이상한 일도 보고되었습니다. CIFAR-10 으로 학습한 흐름이 처음 보는 다른 데이터셋(SVHN)의 이미지에 더 높은 우도를 주었습니다(Nalisnick et al., ICLR 2019. 「Can Images the Flow Never Saw Score Higher?」의 「A known trap」). 그 장의 부분 설명도 같은 모양입니다. 덜 퍼진 데이터가 밀도가 가장 높은 곳 가까이에 놓입니다.
*The inside median (−0.37) is even higher than the common one (−0.61) because the inside points are less spread out than the common ones (noise size 0.05 against 0.08), so they sit closer to the ridge of the moon, the line of highest density. In images something stranger has been reported: a flow trained on CIFAR-10 gave higher likelihood to images from a dataset it had never seen (SVHN) (Nalisnick et al., ICLR 2019; the "A known trap" slide). That slide's partial explanation has the same shape: less spread-out data sits nearer the densest spot.*

## 세 픽셀. 자기회귀 모형의 log p(x) / Three pixels. An autoregressive model's log p(x)
흐름은 사상을 거꾸로 돌려 log p(x) 를 구했습니다. 두 번째 계열인 자기회귀(AR) 모형은 픽셀을 정한 순서로 읽고, 앞 픽셀들이 주어졌을 때 다음 픽셀의 확률을 차례로 곱합니다. 「How Does an AR Model Score Three Pixels?」의 세 픽셀 (1, 0, 1) 을 그대로 씁니다.
*The flow got log p(x) by running its map backward. The second family, autoregressive (AR) models, reads the pixels in a fixed order and multiplies, one after another, the probability of each pixel given the pixels before it. We use the three pixels (1, 0, 1) from that slide.*

| 자리<br>Step | 조건부확률<br>Conditional | 모형이 준 값<br>Model says |
|---|---|---|
| 1 | $p(x_1 = 1)$ | 0.6 |
| 2 | $p(x_2 = 0 \mid x_1 = 1)$ | 0.7 |
| 3 | $p(x_3 = 1 \mid x_1 = 1,\ x_2 = 0)$ | 0.9 |

조건에 들어간 앞 픽셀은 모형이 뽑은 값이 아니라 이 이미지의 참값입니다. 채점할 때는 이미지가 통째로 주어져 있기 때문입니다(「Why Can Training Score Every Pixel at Once?」).
*The earlier pixels in each condition are this image's true values, not the model's own draws: when scoring, the whole image is given.*

**코드를 돌리기 전에** 두 가지를 예측해 보세요.
*Before running, predict two things.*

1. 세 확률을 곱한 $p(1, 0, 1)$ 은? / What is $p(1, 0, 1)$, the product of the three?
2. 그 곱의 로그와, 세 확률의 로그를 하나씩 더한 값은 같을까요? / Is the log of that product the same as adding the three logs one by one?

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   p_true : 자리마다 참값이 나올 확률. 위 표의 「모형이 준 값」 / probability of the true value at each step, the "Model says" column above
#   q      : 지금 자리의 확률, p_true 에서 하나씩 / the probability at the current step, one at a time from p_true
#   prob   : 지금까지 곱한 확률 / the product so far
#   log_p  : 지금까지 더한 log 확률 / the sum of the logs so far
#   loss_i : 이 자리의 손실 −log(참값의 확률). 분류의 교차 엔트로피와 같다
#            this step's loss, −log(probability of the true value); the same as cross-entropy in classification
#
# 참고 / Reference   math.log(q)
#   자연로그 ln q / the natural log ln q.   예 / e.g.   math.log(0.5)  ->  -0.6931

p_true = [0.6, 0.7, 0.9]                       # 자리 1, 2, 3 / steps 1, 2, 3

prob, log_p = 1.0, 0.0                         # 아직 아무것도 곱하지 않았다 / nothing multiplied yet
for i, q in enumerate(p_true, start=1):        # 정한 순서대로 한 자리씩 / one step at a time, in the fixed order
    prob = prob * q                            # 확률은 곱한다 / probabilities multiply
    log_p = log_p + math.log(q)                # 로그로는 더한다 / their logs add
    loss_i = -math.log(q)                      # 이 자리의 손실 / this step's loss
    print(f"자리 / step {i}:  곱 / product = {prob:.3f}   누적 log p / running log p = {log_p:.3f}   손실 / loss = {loss_i:.3f}")
print(f"곱의 로그 / log of the product = {math.log(prob):.3f}")
# 기대값 / Expected   (곱 · 누적 log p · 손실 / product · running log p · loss)
#   step 1:  0.600   -0.511   0.511
#   step 2:  0.420   -0.868   0.357
#   step 3:  0.378   -0.973   0.105
#   곱의 로그 / log of the product = -0.973.  곱의 로그는 로그의 합이라 두 값이 같습니다(슬라이드의 −0.973).
#   The log of a product is the sum of the logs, so the two agree (the slide's −0.973).
#   손실 셋의 합 0.511 + 0.357 + 0.105 = 0.973 = −log p(1, 0, 1). 「Training Is Plain Maximum Likelihood」의 AR 손실이 이것입니다.
#   The three losses add up to 0.973 = −log p(1, 0, 1), the AR loss on that slide.
#   §4 의 TinyAR 은 같은 합을 이미지 한 장의 모든 자리에 대해 마스크로 한 번에 구합니다.
#   §4's TinyAR computes the same sum over every position of an image at once, with a mask.
#   생성은 다릅니다. x1 을 먼저 뽑아야 x2 의 조건이 생기므로 한 자리씩 갑니다(「Masked AR Scoring Is Parallel, Sampling Is Not」).
#   Generating is different: x1 must be drawn before x2 has its condition, so it goes one step at a time.

## §4 (선택) 이진 MNIST 의 작은 자기회귀 모형 / (Optional) A tiny autoregressive model on binary MNIST
위 「세 픽셀」의 계산을 이미지 한 장의 픽셀 D 개로 늘립니다. 이미지 한 장을 0 · 1 픽셀 D 개가 한 줄로 늘어선 것으로 봅니다. 위 행부터, 행 안에서는 왼쪽에서 오른쪽으로 읽습니다. 자기회귀 모형은 연쇄법칙으로 확률을 쪼갭니다(「AR Models Turn the Chain Rule Into a Generator」).
*This extends the three-pixel calculation above to all D pixels of an image. Treat one image as a line of D pixels, each 0 or 1, read row by row from the top, left to right within a row. An autoregressive model splits the probability with the chain rule.*

$$p(x) = p(x_1)\, p(x_2 \mid x_1)\, p(x_3 \mid x_1, x_2) \cdots = \prod_{i=1}^{D} p(x_i \mid x_{<i})$$

가장 작은 판은 조건부확률마다 앞 픽셀들의 가중합에 시그모이드를 씌운 것입니다. $p(x_i = 1 \mid x_{<i}) = \sigma\big(b_i + \sum_{j<i} W_{ij}\, x_j\big)$. 「$j < i$ 인 칸만 쓴다」를 지키는 것이 마스크입니다. W 에 아래삼각 마스크를 곱하면 미래 픽셀의 무게가 0 이 됩니다.
*The smallest version: each conditional is a sigmoid of a weighted sum of the earlier pixels. The mask enforces "use only j < i": multiplying W by a lower-triangular mask zeroes the weights on future pixels.*

「Masked AR Scoring Is Parallel, Sampling Is Not」을 직접 봅니다. 채점은 행렬 곱 한 번으로 모든 자리를 한꺼번에, 생성은 한 자리씩 D 번입니다.
*See that slide for yourself: scoring handles every position at once with one matrix product; sampling goes one position at a time, D times.*

In [ ]:
def binary_rows(images, n):
    """이미지 n 장을 반으로 줄이고(28 × 28 → 14 × 14) 0 · 1 로 바꾼 뒤, 위 행부터 왼쪽에서 오른쪽으로 읽어 한 줄로 편다.
    Halve n images (28 × 28 → 14 × 14), turn them into 0 and 1, then read them row by row, left to right, into one line."""
    x = images[:n].float() / 255.0                         # 밝기 0 ~ 1 / brightness 0 to 1
    x = F.avg_pool2d(x.unsqueeze(1), 2).squeeze(1)         # 2 × 2 칸의 평균으로 크기를 반으로 / average each 2 × 2 block
    return (x > 0.5).float().reshape(n, -1)                # 절반보다 밝으면 1, 행 단위로 한 줄 / 1 if brighter than half, rows into one line

if mnist_train is None:                                    # §0 에서 못 받았으면 한 번 더 / try once more if §0 failed
    try:
        from torchvision import datasets
        mnist_train = datasets.MNIST("data", train=True, download=True)
        mnist_test = datasets.MNIST("data", train=False, download=True)
    except Exception:
        pass

if mnist_train is not None:
    X_ar_train = binary_rows(mnist_train.data, 10000)      # 학습 10,000장 / 10,000 training images
    X_ar_test = binary_rows(mnist_test.data, 1000)         # 보류 1,000장 / 1,000 held-out images
    SIDE = 14
else:                                                      # 대신 라이브러리의 8 × 8 숫자 1,797장 / instead, the library's 1,797 digits of 8 × 8
    from sklearn.datasets import load_digits
    img = torch.tensor(load_digits().images / 16.0, dtype=torch.float32)
    xb = (img > 0.5).float().reshape(len(img), -1)
    X_ar_train, X_ar_test = xb[:1500], xb[1500:]
    SIDE = 8
D = X_ar_train.shape[1]                                    # 이미지 한 장의 픽셀 수 / pixels per image
print("픽셀 수 D / pixels D:", D, "   학습 · 보류 / training, held-out:", len(X_ar_train), len(X_ar_test))
# 기대값 / Expected   196   10000 1000   (MNIST 를 못 받았으면 64   1500 297 / 64   1500 297 if MNIST could not be downloaded)

In [ ]:
class TinyAR(nn.Module):
    """가장 작은 자기회귀 모형. 픽셀 i 가 1 일 확률 = sigmoid(b_i + Σ_{j<i} W_ij x_j).
    The smallest autoregressive model. P(pixel i = 1) = sigmoid(b_i + sum over j < i of W_ij x_j)."""
    def __init__(self, D):
        super().__init__()
        self.W = nn.Parameter(torch.zeros(D, D))           # 앞 픽셀 j 가 픽셀 i 에 주는 무게 / weight from an earlier pixel j to pixel i
        self.b = nn.Parameter(torch.zeros(D))              # 픽셀마다 기본값 / one baseline per pixel
        self.register_buffer("mask", torch.tril(torch.ones(D, D), diagonal=-1))   # j < i 인 칸만 1, 미래를 가린다 / 1 only where j < i; hides the future

    def logits(self, x):
        return x @ (self.W * self.mask).T + self.b         # 모든 자리를 한 번에. 마스크 덕분에 앞 픽셀만 본다 / every position at once; the mask keeps it to earlier pixels

    def log_prob(self, x):
        # 자리마다 log p(x_i | x_<i) 를 더한 것 = 연쇄법칙을 로그로 쓴 것 / the sum over positions of log p(x_i | x_<i): the chain rule in logs
        return -F.binary_cross_entropy_with_logits(self.logits(x), x, reduction="none").sum(dim=1)

    @torch.no_grad()
    def sample(self, n, gen):
        D = self.b.numel()
        x = torch.zeros(n, D)                              # 아직 아무 픽셀도 뽑지 않았다 / no pixel drawn yet
        Wm = self.W * self.mask
        for i in range(D):                                 # 한 자리씩, D 번 / one position at a time, D times
            p = torch.sigmoid(x @ Wm[i] + self.b[i])       # 이미 뽑은 앞 픽셀만 본다 / looks only at pixels already drawn
            x[:, i] = (torch.rand(n, generator=gen) < p).float()
        return x

torch.manual_seed(0)
ar = TinyAR(D)
opt_ar = torch.optim.Adam(ar.parameters(), lr=0.01)
gen_ar = torch.Generator().manual_seed(2)
t0 = time.time()
for step in range(1500):
    idx = torch.randint(0, len(X_ar_train), (200,), generator=gen_ar)
    loss = -ar.log_prob(X_ar_train[idx]).mean()            # 여기도 평균 −log p(x) 하나뿐 / again just the average −log p(x)
    opt_ar.zero_grad(); loss.backward(); opt_ar.step()

with torch.no_grad():
    bits_ar = -ar.log_prob(X_ar_test).mean().item() / (D * math.log(2))      # 픽셀당 비트 / bits per pixel
    on = X_ar_train.mean(dim=0).clamp(1e-3, 1 - 1e-3)                       # 픽셀마다 1 의 비율(앞 픽셀을 안 봄) / each pixel's share of 1s, ignoring earlier pixels
    lp_ind = (X_ar_test * on.log() + (1 - X_ar_test) * (1 - on).log()).sum(dim=1)
    bits_ind = -lp_ind.mean().item() / (D * math.log(2))
print(f"보류 이미지의 픽셀당 비트(낮을수록 좋음) / held-out bits per pixel (lower is better)")
print(f"  자기회귀 / autoregressive         : {bits_ar:.3f}")
print(f"  픽셀마다 따로 / each pixel alone  : {bits_ind:.3f}")
print(f"걸린 시간 / time: {time.time() - t0:.1f} s")
# 기대값 / Expected
#   autoregressive   : 0.257
#   each pixel alone : 0.355
#   (MNIST 를 못 받아 8 × 8 숫자를 썼다면 / with the 8 × 8 digits instead of MNIST: 0.414, 0.541)
#   앞 픽셀을 보는 조건부확률이 같은 이미지를 더 짧게 적습니다. 이산 자료라 비트가 곧 부호 길이입니다(「When Do You Need the Number Itself?」).
#   Conditionals that look at earlier pixels describe the same images in fewer bits. For discrete data, bits are code length.

In [ ]:
# 한 번에 채점한 값 = 한 자리씩 연쇄법칙으로 곱한 값 / One-pass score = the chain rule, one position at a time
x0 = X_ar_test[:1]                                         # 보류 이미지 한 장 / one held-out image
with torch.no_grad():
    one_pass = ar.log_prob(x0).item()                      # 마스크로 모든 자리를 한 번에 / all positions at once with the mask
    step_by_step = 0.0
    Wm = ar.W * ar.mask
    for i in range(D):                                     # 앞 픽셀만 넣어 i 번째 조건부확률을 하나씩 / the i-th conditional from the earlier pixels only
        p1 = torch.sigmoid(x0[0, :i] @ Wm[i, :i] + ar.b[i])
        step_by_step += math.log(p1.item() if x0[0, i] == 1 else 1 - p1.item())
print(f"한 번에 / one pass        : {one_pass:.4f}")
print(f"한 자리씩 / step by step  : {step_by_step:.4f}")
# 기대값 / Expected   두 값이 같습니다 / the two agree:  -25.0507   (8 × 8 숫자라면 / with the 8 × 8 digits: -13.5011)
#   학습 때는 정답 픽셀이 다 있으니 마스크로 한꺼번에 채점해도 한 자리씩 곱한 것과 같습니다.
#   In training every true pixel is known, so scoring all at once with the mask gives the same number as going one by one.

In [ ]:
x_gen_ar = ar.sample(16, torch.Generator().manual_seed(3))   # 한 자리씩 D 번 뽑아 16장 / 16 images, one pixel at a time, D steps
fig, axes = plt.subplots(2, 8, figsize=(8, 2.2))
for a, im in zip(axes.ravel(), x_gen_ar):
    a.imshow(im.reshape(SIDE, SIDE), cmap="Greys", vmin=0, vmax=1); a.axis("off")
fig.suptitle("16 samples from the tiny AR model, drawn one pixel at a time", fontsize=10)
plt.tight_layout(); plt.show()
# 기대값 / Expected   숫자를 닮은 획 조각. 가장 작은 모형이라 또렷한 숫자는 드뭅니다. 우도와 표본 품질은 다른 축입니다
#   (「Exact Likelihood Is Not Sample Quality」).
#   Digit-like stroke fragments; clear digits are rare from the smallest model. Likelihood and sample quality are different axes.

In [ ]:
# 생성 시간이 D 와 함께 늘어나는지 재기. 시간만 재므로 학습 안 한 무작위 가중치로 충분하다.
# Does sampling time grow with D? We only time it, so untrained random weights are enough.
def time_it(fn, repeat=5):
    best = float("inf")
    for _ in range(repeat):                                # 다섯 번 재서 가장 빠른 값 / fastest of five runs
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best * 1000                                     # 밀리초 / milliseconds

print(f"{'D':>5} | {'채점 1번 / score, 1 pass':>26} | {'생성 D번 / sample, D steps':>27}")
for Dk in (49, 196, 784):                                  # 7 × 7, 14 × 14, 28 × 28
    m = TinyAR(Dk)
    with torch.no_grad():
        m.W.normal_(0, 0.1)
    xk = torch.zeros(64, Dk)
    t_score = time_it(lambda: m.log_prob(xk).sum().item())
    t_sample = time_it(lambda: m.sample(64, torch.Generator().manual_seed(0)))
    print(f"{Dk:5d} | {t_score:22.2f} ms | {t_sample:23.1f} ms")
# 기대값 / Expected   (이 노트북을 만든 2코어 CPU, 이미지 64장 / the 2-core CPU used to build this notebook, 64 images)
#      49 | 0.04 ms | 0.9 ms
#     196 | 0.12 ms | 3.5 ms
#     784 | 0.79 ms | 16.7 ms
#   절대 시간은 컴퓨터마다 다릅니다. 볼 것은 늘어나는 모양입니다. 채점은 D 가 커져도 행렬 곱 한 번이고(그 한 번이 조금 무거워질 뿐),
#   생성은 D 번을 차례로 돌아야 해서 D 가 4배가 되면 시간도 4배 넘게 늘어납니다.
#   Absolute times differ by computer; look at the growth. Scoring stays one matrix product as D grows (that one product just gets heavier),
#   while sampling must run D steps in order, so making D four times larger makes it more than four times slower.

## 정리. 예측 다시 읽기 / Wrap-up. Re-read your predictions
1. **몸풀기.** $J = W$, $\det J = 3$ 이라 $p(x) = p(z) / 3$ 입니다. 가중치만 쓰는 직선 사상이라 J 가 어느 점에서나 같습니다.
   *Warm-up. The map uses weights only, a straight-line map, so J is the same at every point.*
2. **§1.** $x_2' = (1.7918,\ 1.5816)$, $\log|\det J| = 0.1$, 되찾을 때는 $x_2 = (x_2' - t) \odot e^{-s}$. 같은 s, t 를 쓰므로 신경망을 거꾸로 돌릴 필요가 없었습니다. 확인 셀에서는 신경망이 s 를 정해 0.1 이 아닌 −0.634 가 나왔고, 행렬로 구한 log-det 과 s 의 합이 같았습니다.
   *§1. The same s and t are reused, so the network never runs backward; in the matrix check the network chose s, so the log-det was −0.634 rather than 0.1, and the matrix gave the same value as the sum of s.*
3. **§3 예측 1.** 흐름 -0.887, 정규분포 하나 -1.911 로 흐름이 높았습니다. 같은 보류 점, 같은 좌표에서 잰 정확한 값이라 바로 비교됩니다.
   *§3 prediction 1. The flow scored higher on the same held-out points in the same coordinates, so the two exact values compare directly.*
4. **§3 예측 2.** 가장 낮은 5%(26개)에 away 는 15 / 15, inside 는 0 / 10 이 들어갔습니다. inside 의 log p(x) 중앙값(-0.37)은 흔한 점의 중앙값(-0.61)보다도 높았습니다(덜 퍼져 능선 가까이에 모였기 때문). 정확한 점수는 흔한 데이터가 없는 곳의 희귀 점은 잡지만, 흔한 데이터와 겹치는 희귀 점은 놓칩니다.
   *§3 prediction 2. The median log p(x) of inside was even higher than that of the common points. The exact score caught the rare points where no common data lives and missed the ones overlapping the common data.*
5. **세 픽셀.** $p(1, 0, 1) = 0.6 \times 0.7 \times 0.9 = 0.378$, $\log p(1, 0, 1) = -0.973$. 곱의 로그는 로그의 합이고, 자리마다의 손실을 더한 0.973 이 곧 $-\log p(1, 0, 1)$ 입니다.
   *Three pixels. The log of the product is the sum of the logs, and the per-step losses add up to $-\log p(1, 0, 1) = 0.973$.*

함정과 짝(「Four Ways to Get This Wrong」): 1번 「log-det 항 빠뜨리기」는 §2 의 빈칸, 2번 「아무 신경망이나 가역이라 여기기」는 §1 확인 셀(가역은 층 구조가 보장), 3번 「AR 의 순서 무시하기」는 세 픽셀의 정한 순서와 §4 의 마스크입니다.
*Pitfalls on the slide: 1 (forgetting the log-det term) is the §2 blank, 2 (thinking any network is invertible) is the §1 check, where the layer structure guarantees invertibility, and 3 (ignoring the ordering in AR) is the fixed order of the three pixels and the mask in §4.*

In [ ]:
# 실행 환경 기록 / Record the runtime
import sys, platform, datetime
print("Runtime :", platform.platform(), "| Python", sys.version.split()[0])
print("torch   :", torch.__version__, "| threads", torch.get_num_threads(), "|", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("Run at  :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))